In [ ]:
!pip install numpy sentence-transformers ripser tqdm requests pandas statsmodels stargazer

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 843.4/843.4 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 5.6 MB/s eta 0:00:00
  Created wheel for hopcroftkarp: filename=hopcroftkarp-1.2.5-py2.py3-none-any.whl size=18104 sha256=8e9f90a11e899f6b160e7cf2087ea16518f8aea83edf1fef7685df6e844830e5
  Stored in directory: /root/.cache/pip/wheels/2a/fd/fe/f4b8fd82894e1d9e04040ef41dc5ae6eb7a8e9b0ef5a9402fe
Successfully built hopcroftkarp


In [ ]:
!pip install transformers==4.51.3 torch accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 162.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 124.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


"Симплициальный комплекс — это топологическое обобщение графа, включающее, помимо вершин и рёбер, симплексы высших размерностей: треугольники (2-симплексы), тетраэдры (3-симплексы) и так далее."

In [ ]:
!pip install vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 8.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.2/248.2 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 MB 8.8 MB/s et

### Генерация трейсов рассуждений

In [ ]:
import os
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from datasets import load_dataset
import torch
import gc
import json
import time
from pathlib import Path
from typing import Any, List
import re
from tqdm import tqdm

def load_math_500_dataset(max_samples=500):
    dataset = load_dataset("HuggingFaceH4/MATH-500")
    data = dataset['test'].select(range(min(max_samples, len(dataset['test']))))
    rows: List[dict[str, Any]] = []
    for entry in data:
        rows.append({
            "id": entry.get("unique_id"),
            "statement": str(entry.get("problem", "")),
            "final_answer": str(entry.get("answer", "")).strip() if entry.get("answer") else None,
            "solution": entry.get("solution"),
        })
    return rows

def load_gsm8k_dataset(max_samples=500):
    dataset = load_dataset("openai/gsm8k", 'main')
    data = dataset['test'].select(range(min(max_samples, len(dataset['test']))))
    rows: List[dict[str, Any]] = []
    id = 0
    for entry in data:
        id += 1
        combined_solution = entry.get("answer")
        final_answer = entry.get("answer").split("####")[1]
        final_answer = str(final_answer).strip() if final_answer is not None else None
        rows.append({
            "id": id,
            "statement": str(entry.get("question", "")),
            "final_answer": final_answer,
            "solution": combined_solution,
        })
    return rows

def load_math_dataset(max_samples=12500):
  dataset = load_dataset("qwedsacf/competition_math")
  data = dataset['train'].select(range(max_samples))
  rows: List[dict[str, Any]] = []
  id = 0
  for entry in data:
      id += 1
      solution = entry.get("solution")
      final_answer = extract_answer_from_text(solution)
      rows.append({
          "id": id,
          "statement": str(entry.get("problem", "")),
          "final_answer": final_answer,
          "solution": entry.get("solution"),
      })
  return rows

def ensure_dir(path: str | Path) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)

def extract_answer_from_text(text: str) -> str:
    if not text:
        return 'invalid'

    pattern_boxed = r"\\boxed\{(.*)\}"
    match_boxed = re.search(pattern_boxed, text.strip())
    if match_boxed:
        answer = match_boxed.group(1).strip()
        answer = re.sub(r'[.\n]+$', '', answer)
        return answer
    pattern_answer = r"(?i)(?:final\s+)?answer\s*:\s*([^\n]+)"
    match_answer = re.search(pattern_answer, text.strip())
    if match_answer:
        answer = match_answer.group(1).strip()
        boxed_in_answer = re.search(r"\\?\\?boxed\s*\{\s*([^}]+?)\s*\}", answer)
        if boxed_in_answer:
            answer = boxed_in_answer.group(1).strip()
        answer = re.sub(r'[.\n]+$', '', answer)
        return answer

    return 'invalid'


def setup_vllm_model(model_name: str, quantization: str = "awq_marlin",
                     gpu_memory_utilization: float = 0.85,
                     max_model_len: int = 3800):

    print(f"Loading {model_name} with vLLM...")

    llm = LLM(
        model=model_name,
        tensor_parallel_size=1,
        trust_remote_code=True,
        max_model_len=max_model_len,
        dtype="float16",
        quantization=quantization,
        gpu_memory_utilization=gpu_memory_utilization,
        disable_log_stats=True,
        enforce_eager=False
    )

    tokenizer = llm.get_tokenizer()
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"Model loaded successfully")
    return llm, tokenizer

def generate_batch_vllm(
    llm: LLM,
    prompts: List[str],
    max_tokens: int = 3800,
    temperature: float = 0.0
) -> List[str]:

    sampling_params = SamplingParams(
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=1.0,
        top_k=-1,
        stop=None,
        repetition_penalty=1.0
    )

    outputs = llm.generate(prompts, sampling_params)
    responses = [output.outputs[0].text for output in outputs]

    return responses

def prepare_prompts(rows: List[dict], tokenizer, sys_prompt: str) -> List[str]:
    prompts = []
    for ex in rows:
        statement = ex.get("statement", "")
        prompt = f"{sys_prompt}Problem:\n{statement}\n\nSolution:"
        prompts.append(prompt)
    return prompts

def main():

    model_name = "Qwen/Qwen2.5-1.5B"
    judge_model_name = "Qwen/Qwen2.5-14B-Instruct-AWQ"

    max_new_tokens = 3800
    max_samples = 12500
    batch_size = 16
    judge_batch_size = 64

    out = "data/math_traces/traces_math_qwen1.5b_vllm_full.jsonl"
    ensure_dir(out)

    print("=" * 60)
    print(f"Loading dataset (max {max_samples} samples)...")
    print("=" * 60)
    rows = load_math_dataset(max_samples)
    sys_prompt = """You are an expert competition mathematician.
    Solve the problem carefully and present a clear, rigorous step-by-step solution.
    Ensure each step is justified and consistent.
    End with the final line formatted exactly as 'Answer: <answer>' where <answer> is the answer to the problem.
    """

    print("\n" + "=" * 60)
    print("Setting up main model with vLLM...")
    print("=" * 60)
    model, tokenizer = setup_vllm_model(model_name, quantization = None, gpu_memory_utilization=0.7)

    print("\n" + "=" * 60)
    print("Preparing prompts...")
    print("=" * 60)
    all_prompts = prepare_prompts(rows, tokenizer, sys_prompt)

    print("\n" + "=" * 60)
    print(f"Generating responses in batches of {batch_size}...")
    print("=" * 60)

    all_responses = []
    all_traces = []

    start_total = time.time()

    for batch_start in tqdm(range(0, len(all_prompts), batch_size), desc="Generating batches"):
        batch_end = min(batch_start + batch_size, len(all_prompts))
        batch_prompts = all_prompts[batch_start:batch_end]
        batch_rows = rows[batch_start:batch_end]

        batch_start_time = time.time()
        try:
            responses = generate_batch_vllm(
                model,
                batch_prompts,
                max_tokens=max_new_tokens,
                temperature=0.0
            )
            gen_time = time.time() - batch_start_time

            for idx, (ex, response) in enumerate(zip(batch_rows, responses)):

                pred_answer = extract_answer_from_text(response)
                true_answer = ex.get("final_answer")


                print(f"\nProblem {batch_start + idx + 1}:")
                print(f"Predicted answer: {pred_answer}")
                print(f"True answer: {true_answer}")

                trace = {
                    "id": ex.get("id"),
                    "statement": ex.get("statement"),
                    "model": model_name,
                    "trace": response,
                    "pred_answer": pred_answer,
                    "final_answer": true_answer,
                    "reference_solution": ex.get("solution"),
                    "is_correct": None,
                    "timestamp": int(time.time()),
                    "processing_time": gen_time / len(batch_prompts)
                }
                all_traces.append(trace)
                all_responses.append((pred_answer, true_answer))
                print(trace)

                with open(out, "a", encoding="utf-8") as out_f:
                    out_f.write(json.dumps(trace, ensure_ascii=False) + "\n")
                    out_f.flush()

        except Exception as e:
            print(f"Batch generation error: {e}")
            for ex in batch_rows:
                trace = {
                    "id": ex.get("id"),
                    "statement": ex.get("statement"),
                    "model": model_name,
                    "trace": f"ERROR: {str(e)}",
                    "pred_answer": 'invalid',
                    "final_answer": ex.get("final_answer"),
                    "reference_solution": ex.get("solution"),
                    "is_correct": 'No',
                    "timestamp": int(time.time()),
                    "processing_time": 0
                }
                all_traces.append(trace)
                all_responses.append(('invalid', ex.get("final_answer")))

        if batch_start % (batch_size * 4) == 0:
            print(f"\nGPU memory after batch {batch_start//batch_size + 1}:")
            for i in range(torch.cuda.device_count()):
                allocated = torch.cuda.memory_allocated(i) / 1e9
                print(f"  GPU {i}: {allocated:.2f}GB allocated")



    total_gen_time = time.time() - start_total

    print("\n" + "=" * 60)
    print("FINAL STATISTICS")
    print("=" * 60)

    total_valid = sum(1 for t in all_traces if t['final_answer'] is not None and t['pred_answer'] != 'invalid')

    print(f"Total problems: {len(all_traces)}")
    print(f"Valid problems (with answers): {total_valid}")

    print(f"\nGeneration speed: {len(all_traces)/total_gen_time:.2f} samples/sec")
    print(f"\nResults saved to: {out}")

if __name__ == "__main__":
    main()

In [ ]:
rows = load_math_dataset(12500)

### Оценка ответов моделью-судьей

In [ ]:
traces_path = "/content/traces_math_qwen1.5b_vllm_full.jsonl"
final_out = "/content/traces_math_qwen1.5b_vllm_final.jsonl"

In [ ]:
import json

from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from datasets import load_dataset
import torch
import gc
import json
import time
from pathlib import Path
from typing import Any, List
import re
from tqdm import tqdm

def llm_judge_batch(
    true_responses: List[str],
    model_responses: List[str],
    judge_llm: LLM,
    judge_tokenizer,
    batch_size: int = 32
) -> List[str]:
    prompts = []
    for true_resp, model_resp in zip(true_responses, model_responses):
        if model_resp == 'invalid' or true_resp is None:
            prompts.append(None)
            continue

        judge_prompt = f"""
        Look at the following two expressions (answers to a math problem) and judge whether they are equivalent. Only perform trivial simplifications

        Examples:

            Expression 1: $2x+3$
            Expression 2: $3+2x$

        Yes

            Expression 1: 3/2
            Expression 2: 1.5

        Yes

            Expression 1: $x^2+2x+1$
            Expression 2: $y^2+2y+1$

        No

            Expression 1: $x^2+2x+1$
            Expression 2: $(x+1)^2$

        Yes

            Expression 1: 3245/5
            Expression 2: 649

        No
        (these are actually equal, don't mark them equivalent if you need to do nontrivial simplifications)

            Expression 1: \text{{tanya}}
            Expression 2: tanya

        Yes

            Expression 1: 2/(-3)
            Expression 2: -2/3

        Yes
        (trivial simplifications are allowed)

            Expression 1: 72 degrees
            Expression 2: 72

        Yes
        (give benefit of the doubt to units)

            Expression 1: 64
            Expression 2: 64 square feet

        Yes
        (give benefit of the doubt to units)

        ---

        YOUR TASK


        Respond with only "Yes" or "No" (without quotes). Do not include a rationale.

            Expression 1: {true_resp}
            Expression 2: {model_resp}
        """.strip()

        messages = [{"role": "user", "content": judge_prompt}]
        prompt = judge_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        prompts.append(prompt)

    results = []
    valid_indices = [i for i, p in enumerate(prompts) if p is not None]
    valid_prompts = [p for p in prompts if p is not None]

    if not valid_prompts:
        return ['No'] * len(prompts)

    sampling_params = SamplingParams(
        max_tokens=10,
        temperature=0.0,
        top_p=1.0,
        top_k=-1
    )

    for i in range(0, len(valid_prompts), batch_size):
        batch_prompts = valid_prompts[i:i+batch_size]
        outputs = judge_llm.generate(batch_prompts, sampling_params)

        for output in outputs:
            response = output.outputs[0].text.strip().lower()
            results.append('Yes' if 'yes' in response else 'No')

    full_results = ['No'] * len(prompts)
    for idx, res in zip(valid_indices, results):
        full_results[idx] = res

    return full_results

def setup_vllm_model(model_name: str, gpu_memory_utilization: float = 0.85,
                     max_model_len: int = 3800):

    print(f"Loading {model_name} with vLLM...")

    llm = LLM(
        model=model_name,
        tensor_parallel_size=1,
        trust_remote_code=True,
        max_model_len=max_model_len,
        dtype="float16",
        quantization="awq_marlin",
        gpu_memory_utilization=gpu_memory_utilization,
        disable_log_stats=True,
        enforce_eager=False
    )

    tokenizer = llm.get_tokenizer()
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print(f"Model loaded successfully")
    return llm, tokenizer


all_traces = []

with open(traces_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            trace =  json.loads(line)
            all_traces.append(trace)

judge_model_name = "Qwen/Qwen2.5-14B-Instruct-AWQ"
judge_batch_size = 64

print("\n" + "=" * 60)
print("Setting up judge model with vLLM...")
print("=" * 60)
judge_model, judge_tokenizer = setup_vllm_model(judge_model_name, gpu_memory_utilization=0.3)

print("\n" + "=" * 60)
print("Judging all answers...")
print("=" * 60)

true_answers = [t["final_answer"] for t in all_traces]
pred_answers = [t["pred_answer"] for t in all_traces]

judge_start = time.time()
judge_results = llm_judge_batch(
    true_answers,
    pred_answers,
    judge_model,
    judge_tokenizer,
    batch_size=judge_batch_size
)
judge_time = time.time() - judge_start
print(f"Judging complete in {judge_time:.2f}s")

print("\n" + "=" * 60)
print("Updating traces with judge results...")
print("=" * 60)

with open(final_out, "w", encoding="utf-8") as out_f:
    for trace, is_correct in zip(all_traces, judge_results):
        trace["is_correct"] = is_correct
        out_f.write(json.dumps(trace, ensure_ascii=False) + "\n")

print("\n" + "=" * 60)
print("FINAL STATISTICS")
print("=" * 60)

correct_count = sum(1 for r in judge_results if r == 'Yes')
total_valid = sum(1 for t in all_traces if t['final_answer'] is not None and t['pred_answer'] != 'invalid')

print(f"Total problems: {len(all_traces)}")
print(f"Valid problems (with answers): {total_valid}")
print(f"Correct predictions: {correct_count}")
if total_valid > 0:
    print(f"Accuracy: {correct_count/total_valid*100:.1f}%")

print(f"Judging speed: {len(all_traces)/judge_time:.2f} samples/sec")
print(f"\nResults saved to: {final_out}")

In [ ]:
import re

In [ ]:
def extract_answer_from_text(text: str) -> str:
    if not text:
        return 'invalid'

    pattern_boxed = r"\\boxed\{(.*)\}"
    match_boxed = re.search(pattern_boxed, text.strip())
    if match_boxed:
        answer = match_boxed.group(1).strip()
        answer = re.sub(r'[.\n]+$', '', answer)
        return answer

    pattern_answer = r"(?i)(?:final\s+)?answer\s*:\s*([^\n]+)"
    match_answer = re.search(pattern_answer, text.strip())
    if match_answer:
        answer = match_answer.group(1).strip()
        boxed_in_answer = re.search(r"\\?\\?boxed\s*\{\s*([^}]+?)\s*\}", answer)
        if boxed_in_answer:
            answer = boxed_in_answer.group(1).strip()
        answer = re.sub(r'[.\n]+$', '', answer)
        return answer

    return 'invalid'

def normalize_answer(answer: str) -> str:
    answer = answer.strip().lower()
    answer = re.sub(r'[,$% ]', '', answer)
    return answer

response = r"""To convert the point \((0, 3)\) from rectangular coordinates to polar coordinates, we need to find the values of \(r\) and \(\theta\).
The formulas for converting from rectangular coordinates \((x, y)\) to polar coordinates \((r, \theta)\) are:

\[ r = \sqrt{x^2 + y^2} \]
\[ \theta = \tan^{-1}\left(\frac{y}{x}\right) \]

Given the point \((0, 3)\):

1. **Calculate \(r\):**
   \[
   r = \sqrt{x^2 + y^2} = \sqrt{0^2 + 3^2} = \sqrt{9} = 3
   \]

2. **Calculate \(\theta\):**
   Since \(x = 0\) and \(y = 3\), we have:
   \[
   \theta = \tan^{-1}\left(\frac{y}{x}\right) = \tan^{-1}\left(\frac{3}{0}\right)
   \]
   However, \(\tan^{-1}\left(\frac{3}{0}\right)\) is undefined because division by zero is not allowed.
   Instead, we recognize that when \(x = 0\) and \(y > 0\), the angle \(\theta\) is \(\frac{\pi}{2}\).

Therefore, the polar coordinates of the point \((0, 3)\) are:
\[
(r, \theta) = (3, \frac{\pi}{2})
\]

Final answer:
\[
\boxed{(3, \frac{\pi}{2})}
\]"""

extracted = extract_answer_from_text((response))
normalized = normalize_answer(extracted)

print(f"Extracted: '{extracted}'")
print(f"Normalized: '{normalized}'")

Extracted: '(3, \frac{\pi}{2})'
Normalized: '(3\frac{\pi}{2})'


### Построение кластеризационного графа

In [ ]:
import re

def _strip_math_inline_markers(text: str) -> str:
    return text.replace("\\(", "").replace("\\)", "").replace("$", "").replace("\\[","").replace("\\]","").replace("\\","").replace("<think>","")

def segment_steps(text: str, min_len: int = 2) -> list[str]:
    if not text:
        return []
    cleaned = _strip_math_inline_markers(text)
    segments: list[str] = []
    for ln in cleaned.splitlines():
        ln = ln.strip()
        if not ln:
            continue
        pieces = re.split(r"(?<=\.)\s+", ln)
        segments.extend(piece.strip() for piece in pieces if piece.strip())
    return segments

Модель для создания эмбеддингов шагов

In [ ]:
from dataclasses import dataclass
from typing import Iterable, Optional
from sentence_transformers import SentenceTransformer
import numpy as np


@dataclass
class EmbeddingConfig:
    model_name: str = "sentence-transformers/all-mpnet-base-v2"
    batch_size: int = 32
    device: Optional[str] = None


class SentenceTransformerEmbedder:
    def __init__(self, cfg: EmbeddingConfig):
        self.cfg = cfg or EmbeddingConfig()
        self.model = SentenceTransformer(self.cfg.model_name, device=self.cfg.device)

    def encode(self, texts: Iterable[str]) -> np.ndarray:
        arr = self.model.encode(
            list(texts),
            batch_size=self.cfg.batch_size,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True,
        )
        return np.asarray(arr)


def cosine_similarity_matrix(X: np.ndarray) -> np.ndarray:
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
    return Xn @ Xn.T

Подсчет alignment score из статьи shape of reasoning

In [ ]:
from dataclasses import dataclass

import numpy as np


@dataclass
class AlignmentResult:
    indices: list[tuple[int, int]]
    score: float
    coverage: float


def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    an = a / (np.linalg.norm(a) + 1e-12)
    bn = b / (np.linalg.norm(b) + 1e-12)
    return float(np.clip(an @ bn, -1.0, 1.0))


def align_steps(
    model_emb: np.ndarray,
    gold_emb: np.ndarray,
    gap_penalty: float = 0.0,
) -> AlignmentResult:

    n, m = model_emb.shape[0], gold_emb.shape[0]
    S = np.zeros((n, m), dtype=float)
    for i in range(n):
        for j in range(m):
            S[i, j] = cosine_sim(model_emb[i], gold_emb[j])

    dp = np.full((n + 1, m + 1), -1e9, dtype=float)
    bt = np.zeros((n + 1, m + 1, 2), dtype=int)
    dp[0, 0] = 0.0
    for i in range(n + 1):
        for j in range(m + 1):
            if i < n and j < m:
                v = dp[i, j] + S[i, j]
                if v > dp[i + 1, j + 1]:
                    dp[i + 1, j + 1] = v
                    bt[i + 1, j + 1] = (i, j)
            if i < n:
                v = dp[i, j] - gap_penalty
                if v > dp[i + 1, j]:
                    dp[i + 1, j] = v
                    bt[i + 1, j] = (i, j)
            if j < m:
                v = dp[i, j] - gap_penalty
                if v > dp[i, j + 1]:
                    dp[i, j + 1] = v
                    bt[i, j + 1] = (i, j)

    i, j = n, m
    pairs: list[tuple[int, int]] = []
    while i > 0 or j > 0:
        pi, pj = bt[i, j]
        if pi == i - 1 and pj == j - 1:
            pairs.append((i - 1, j - 1))
        i, j = pi, pj
    pairs.reverse()

    sim_sum = 0.0
    for (i, j) in pairs:
        sim_sum += S[i, j]
    avg_sim = sim_sum / max(1, len(pairs))
    gold_covered = len(set(j for _, j in pairs)) / max(1, m)
    return AlignmentResult(indices=pairs, score=avg_sim, coverage=gold_covered)


Сегментация рассуждения на шаги и получение эмбеддингов шагов

In [ ]:
import sys
from pathlib import Path as _Path

import argparse
import json
from pathlib import Path
from typing import Any, Iterable

import numpy as np
from tqdm import tqdm


def read_jsonl(path: str | Path) -> Iterable[dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)


def ensure_dir(path: str | Path) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)


def save_npz(path: str | Path, **arrays: np.ndarray) -> None:
    ensure_dir(path)
    np.savez_compressed(path, allow_pickle=True, **arrays)


def main() -> None:

    out = "data/math500_align_dp/deepseek-ai/DeepSeek-R1-Distill-Qwen-32B/align_math500_DeepSeek32B.jsonl"
    embed_outdir = "data/math500_embed/deepseek-ai/DeepSeek-R1-Distill-Qwen-32B/"
    model_name = "sentence-transformers/all-mpnet-base-v2"
    device = "cuda"
    traces = "/content/traces_math500_deepseek32b_vllm_final.jsonl"

    ensure_dir(out)

    embed_root = Path(embed_outdir)
    trace_embed_dir = embed_root / "trace"
    gold_embed_dir = embed_root / "gold"
    trace_embed_dir.mkdir(parents=True, exist_ok=True)
    gold_embed_dir.mkdir(parents=True, exist_ok=True)

    embedder = SentenceTransformerEmbedder(
        EmbeddingConfig(model_name=model_name, device=device)
    )

    out_f = open(out, "w", encoding="utf-8")
    n = 0
    for tr in tqdm(read_jsonl(traces), desc="Aligning traces", unit="trace"):
        pid = str(tr.get("id")).replace('/', '_')
        print(tr.get("trace"))
        trace_steps = segment_steps(tr.get("trace"))
        print(trace_steps)
        gold_steps = segment_steps(tr.get("reference_solution"))

        X_model = embedder.encode(trace_steps)
        X_gold = embedder.encode(gold_steps)
        save_npz(trace_embed_dir / f"{pid}.npz", X=X_model)
        save_npz(gold_embed_dir / f"{pid}.npz", X=X_gold)
        align = align_steps(X_model, X_gold)
        row = {
            "id": pid,
            "indices": [(int(i), int(j)) for i, j in align.indices],
            "score": float(align.score),
            "coverage": float(align.coverage),
        }
        out_f.write(json.dumps(row) + "\n")
        n += 1
    out_f.close()
    print(
        "Wrote "
        f"{n} alignments to {out} with embeddings under {trace_embed_dir.parent}"
    )


if __name__ == "__main__":
    main()

In [ ]:
from typing import Literal, Optional
from ripser import ripser
import numpy as np
from scipy.stats import skew
from persim import PersLandscapeExact

def vr_diagrams(
    X: np.ndarray,
    metric: Literal["euclidean", "cosine"] = "cosine",
    maxdim: int = 1,
    thresh: Optional[float] = None,
) -> dict[int, np.ndarray]:
    out = ripser(X, maxdim=maxdim, metric=metric, thresh=thresh if thresh is not None else float("inf"))
    return {i: dgms for i, dgms in enumerate(out["dgms"])}

def vr_features(
    diagrams: dict[str, np.ndarray],
) -> dict[str, float]:
    feats: dict[str, float] = {}
    for dim, dgm in diagrams.items():
        if dgm.size == 0:
            feats.update({
                f"{dim}_count": 0.0,
                f"{dim}_total_life": 0.0,
                f"{dim}_max_life": 0.0,
                f"{dim}_mean_life": 0.0,
                f"{dim}_entropy": 0.0,
                f"{dim}_skewness": 0.0,
                f"{dim}_max_birth": 0.0,
                f"{dim}_max_death": 0.0,
            })
            continue
        births = dgm[:, 0]
        deaths = dgm[:, 1]
        finite = np.isfinite(deaths)
        life = np.where(np.isfinite(deaths), deaths - births, 0.0)
        total_life = float(np.sum(life))
        max_life = float(np.max(life) if life.size else 0.0)
        finite = np.isfinite(deaths)
        if total_life > 0.0:
            positive = life > 0.0
            p = (life[positive] / total_life).astype(float)
            entropy = float(-np.sum(p * np.log(p))) if p.size else 0.0
        else:
            entropy = 0.0
        skewness = float(skew(life)) if life.size > 2 else 0.0
        feats.update({
            f"{dim}_count": float(dgm.shape[0]),
            f"{dim}_total_life": total_life,
            f"{dim}_max_life": max_life,
            f"{dim}_mean_life": total_life / float(finite.shape[0]) if finite.shape[0] > 0 else 0.0,
            f"{dim}_entropy": entropy,
            f"{dim}_skewness": skewness,
            f"{dim}_max_birth": float(np.max(births) if births.size else 0.0),
            f"{dim}_max_death": float(np.max(deaths[finite]) if np.any(finite) else 0.0),
        })
    return feats

def landscape_features(
    diagrams: dict[str, np.ndarray],
) -> dict[str, float]:

    feats: dict[str, float] = {}
    dgms = [d for d in diagrams.values()]
    for dim, dgm in diagrams.items():
        if dgm.size == 0:
            feats.update({
                f"{dim}_landscape_mean": 0.0,
                f"{dim}_landscape_max": 0.0,
                f"{dim}_landscape_area": 0.0,
            })
            continue
        dim = int(dim[-1])
        pl = PersLandscapeExact(dgms=dgms, hom_deg=dim)
        landscape = pl.critical_pairs
        xs = np.array([x for x, _ in landscape[0]])
        ys = np.array([y for _, y in landscape[0]])
        full_xs = [np.array([x for x, _ in level]) for level in landscape]
        full_ys = [np.array([y for _, y in level]) for level in landscape]
        mean_val = float(np.mean(ys)) if ys.size > 0 else 0.0
        max_val = float(np.max(ys)) if ys.size > 0 else 0.0
        area_val = float(np.trapezoid(ys, xs)) if ys.size > 1 else 0.0
        total_area_val = float(sum(np.trapezoid(y, x) for x, y in zip(full_xs, full_ys) if y.size > 1)) if ys.size > 1 else 0.0
        feats.update({
            f"H{dim}_landscape_mean": mean_val,
            f"H{dim}_landscape_max": max_val,
            f"H{dim}_landscape_area": area_val,
        })
    return feats

In [ ]:
import numpy as np

def betti_curve(dgm: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    if dgm.size == 0:
        return np.array([]), np.array([])
    ts = []
    tracker = []
    for birth, death in dgm:
        tracker.append((birth, +1))
        tracker.append((death, -1))
    tracker.sort(key=lambda x: x[0])
    counts = []
    count = 0
    for t, delta in tracker:
        if not ts or t != ts[-1]:
            if np.isfinite(t):
                ts.append(t)
                counts.append(count)
        count += delta
    return np.array(ts), np.array(counts, dtype=float)


def betti_features(
    diagrams: dict[str, np.ndarray]
) -> dict[str, float]:
    out = {
        "H0_betti_peak": 0.0,
        "H0_betti_location": 0.0,
        "H0_betti_width": 0.0,
        "H0_betti_centroid": 0.0,
        "H0_betti_spread": 0.0,
        "H0_betti_trend": 0.0,
        "H1_betti_peak": 0.0,
        "H1_betti_location": 0.0,
        "H1_betti_width": 0.0,
        "H1_betti_centroid": 0.0,
        "H1_betti_spread": 0.0,
        "H1_betti_trend": 0.0,
    }
    for dim, dgm in diagrams.items():
        if dgm.size == 0:
            continue
        deaths = dgm[:, 1]
        finite = np.isfinite(deaths)
        if not finite.any():
            continue

        ts, bc = betti_curve(dgm)
        t_min, t_max = np.min(ts), np.max(ts)

        if bc.size != ts.size or not np.isfinite(bc).all():
            continue
        peak = float(np.max(bc)) if bc.size else 0.0
        if peak <= 0.0:
            out[f"H{dim}_betti_peak"] = 0.0
            continue

        i_peak = int(np.argmax(bc))
        t_range = t_max - t_min
        t_peak_n = float((ts[i_peak] - t_min) / t_range)

        mass = float(np.trapezoid(bc, ts))
        if mass > 0:
            centroid = float(np.trapezoid(bc * ts, ts) / mass)
            var = float(np.trapezoid(bc * (ts - centroid) ** 2, ts) / mass)
            spread = float(np.sqrt(max(var, 0.0)))
            centroid_n = float((centroid - t_min) / t_range)
            spread_n = float(spread / t_range)
        else:
            centroid_n = 0.0
            spread_n = 0.0

        half = 0.5 * peak
        mask = bc >= half
        if np.any(mask):
            i0 = int(np.argmax(mask))
            i1 = int(len(bc) - 1 - np.argmax(mask[::-1]))
            width_n = float((ts[i1] - ts[i0]) / t_range) if i1 >= i0 else 0.0
        else:
            width_n = 0.0

        bc_std = float(bc.std())
        ts_std = float(ts.std())
        if bc_std > 0 and ts_std > 0:
            trend = float(np.corrcoef(ts, bc)[0, 1])
            if not np.isfinite(trend):
                trend = 0.0
        else:
            trend = 0.0

        out[f"{dim}_betti_peak"] = peak
        out[f"{dim}_betti_location"] = t_peak_n
        out[f"{dim}_betti_width"] = width_n
        out[f"{dim}_betti_centroid"] = centroid_n
        out[f"{dim}_betti_spread"] = spread_n
        out[f"{dim}_betti_trend"] = trend
    return out

Подсчет топологических фичей на пространстве эмбеддингов

In [ ]:
import argparse
import json
from pathlib import Path
from typing import IO, Tuple
import sys

import numpy as np
from tqdm import tqdm


def ensure_dir(path: str | Path) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)


def load_npz(path: str | Path) -> dict[str, np.ndarray]:
    with np.load(path) as data:
        return {k: data[k] for k in data.files}


def save_npz(path: str | Path, **arrays: np.ndarray) -> None:
    ensure_dir(path)
    np.savez_compressed(path, allow_pickle=True, **arrays)


def compute_diagrams(
    embeddings: np.ndarray,
    metric: str,
    maxdim: int,
) -> dict[str, np.ndarray]:
    diagrams = vr_diagrams(embeddings, metric=metric, maxdim=maxdim)
    return {f"H{dim}": dgm for dim, dgm in diagrams.items()}


def compute_feature_row(
    pid: str,
    diagrams: dict[str, np.ndarray],
) -> dict[str, object]:
    feats: dict[str, object] = {}
    feats.update(vr_features(diagrams))
    feats.update(betti_features(diagrams))
    feats.update(landscape_features(diagrams))
    serialisable = {
        key: (value.tolist() if isinstance(value, np.ndarray) else value)
        for key, value in feats.items()
    }
    return {"id": pid, **serialisable}


def embeddings_to_diagrams_and_features(
    emb_dir: Path,
    diagram_out_dir: Path,
    feature_writer: IO[str],
    metric: str,
    maxdim: int,
    reuse_diagrams: bool = False,
) -> Tuple[int, int, int]:
    if not emb_dir.exists():
        raise FileNotFoundError(f"Embeddings directory '{emb_dir}' not found")

    diagram_out_dir.mkdir(parents=True, exist_ok=True)

    diag_written = 0
    diag_reused = 0
    feat_count = 0

    for npz_path in tqdm(sorted(emb_dir.glob("*.npz")), desc="Computing TDA", unit="file"):
        pid = npz_path.stem
        diag_path = diagram_out_dir / f"{pid}.npz"

        diagrams: dict[str, np.ndarray]
        if reuse_diagrams and diag_path.exists():
            diagrams = load_npz(diag_path)
            diag_reused += 1
        else:
            arrays = load_npz(npz_path)
            X = arrays.get("X")
            print(X.shape)
            if X is None or X.shape[0] < 2:
                continue
            diagrams = compute_diagrams(X, metric=metric, maxdim=maxdim)
            save_npz(diag_path, **diagrams)
            diag_written += 1

        row = compute_feature_row(pid, diagrams)
        feature_writer.write(json.dumps(row) + "\n")
        feat_count += 1

    return diag_written, diag_reused, feat_count


def diagrams_dir_to_features(
    diag_dir: Path,
    feature_writer: IO[str],
    desc: str = "Computing TDA features",
) -> int:
    if not diag_dir.exists():
        raise FileNotFoundError(f"Diagram directory '{diag_dir}' not found")

    feature_count = 0
    for npz_path in tqdm(sorted(diag_dir.glob("*.npz")), desc=desc, unit="file"):
        pid = npz_path.stem
        diagrams = load_npz(npz_path)
        row = compute_feature_row(pid, diagrams)
        feature_writer.write(json.dumps(row) + "\n")
        feature_count += 1
    return feature_count


def main() -> None:

    model = "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B"

    split = "trace"
    emb_dir = "/content/data/math500_embed/deepseek-ai/DeepSeek-R1-Distill-Qwen-32B/trace"

    embed_root = "data/math_embed"

    diag_root = "data/math_tda_diags"

    features_root = "data/math_tda"

    diag_dir = None
    features_out = None
    metric = "cosine"
    maxdim = 1
    reuse_diagrams=False


    model_slug = model
    split = split

    emb_dir = Path(emb_dir) if emb_dir else Path(embed_root) / model_slug / split
    diag_dir = Path(diag_dir) if diag_dir else Path(diag_root) / model_slug / split
    if features_out:
        features_out = Path(features_out)
    else:
        features_out = Path(features_root) / split / f"{model_slug}.jsonl"

    ensure_dir(features_out)

    with open(features_out, "w", encoding="utf-8") as feature_writer:
        diag_written, diag_reused, feat_count = embeddings_to_diagrams_and_features(
            emb_dir=emb_dir,
            diagram_out_dir=diag_dir,
            feature_writer=feature_writer,
            metric=metric,
            maxdim=maxdim,
            reuse_diagrams=reuse_diagrams,
        )

    total_diagrams = diag_written + diag_reused
    if reuse_diagrams and diag_reused > 0:
        print(
            f"Processed {split} diagrams for {total_diagrams} items "
            f"({diag_written} recomputed, {diag_reused} reused) in {diag_dir}"
        )
    else:
        print(f"Wrote {split} diagrams for {diag_written} items to {diag_dir}")

    print(f"Wrote {split} features for {feat_count} items to {features_out}")


if __name__ == "__main__":
    main()

Построение графа и подсчет графовых признаков с помощью библиотеки networkx

In [ ]:
from collections import defaultdict, deque
import heapq
import math
import numpy as np
from sklearn.cluster import KMeans
import networkx as nx

def build_global_clusters(all_embeddings: np.ndarray, n_clusters: int = 200):
    kmeans = KMeans(n_clusters=n_clusters, random_state=0).fit(all_embeddings)
    return kmeans

def build_graph(X: np.ndarray, kmeans):
    X = X.astype(np.float64)
    labels = kmeans.predict(X)
    centers = kmeans.cluster_centers_

    cluster_embeddings = defaultdict(list)
    for idx, label in enumerate(labels):
        cluster_embeddings[label].append(X[idx])

    node_avg_embeddings = {}
    for cluster_id, embeddings_list in cluster_embeddings.items():
        node_avg_embeddings[cluster_id] = np.mean(embeddings_list, axis=0)

    path = [int(label) for label in labels]
    print(path)
    distances = [float(np.linalg.norm(centers[i] - centers[i-1])) for i in path[1:]]
    return path, distances, node_avg_embeddings


def analyze_graph_simple(path, distances, node_avg_embeddings):
    adj = defaultdict(list)
    for u, v, w in zip(path, path[1:], distances):
        if u != v:
            adj[u].append((v, w))

    G = nx.DiGraph()
    for u, neighbors in adj.items():
      for v, w in neighbors:
        G.add_edge(u, v, weight=w)

    for node_id, avg_emb in node_avg_embeddings.items():
        G.nodes[node_id]['embedding'] = avg_emb.tolist()

    G_und = G.to_undirected()

    n_nodes = G_und.number_of_nodes()
    n_edges = G_und.number_of_edges()

    degree_centrality = nx.degree_centrality(G_und)

    metrics = {}

    metrics['n_nodes'] = n_nodes
    metrics['n_edges'] = n_edges
    metrics['path_length'] = len(path)
    metrics['n_unique_clusters'] = len(set(path))
    metrics['density'] = nx.density(G_und) if n_nodes > 0 else 0
    metrics['edge_to_node_ratio'] = n_edges / n_nodes if n_nodes > 0 else 0

    metrics['is_connected'] = int(nx.is_connected(G_und) if n_nodes > 0 else False)
    metrics['is_strongly_connected'] = int(nx.is_strongly_connected(G) if n_nodes > 0 else False)
    metrics['n_connected_components'] = nx.number_connected_components(G_und) if n_nodes > 0 else 0
    metrics['n_strongly_connected'] = nx.number_strongly_connected_components(G) if n_nodes > 0 else 0

    if n_nodes > 0:
        component_sizes = [len(c) for c in nx.connected_components(G_und)]
        metrics['largest_component_size'] = max(component_sizes) if component_sizes else 0
    else:
        metrics['largest_component_size'] = 0

    if n_nodes > 0:
        degree_seq = [d for n, d in G_und.degree()]
        in_degree_seq = [d for n, d in G.in_degree()]
        out_degree_seq = [d for n, d in G.out_degree()]

        metrics['avg_degree'] = np.mean(degree_seq)
        metrics['median_degree'] = np.median(degree_seq)
        metrics['std_degree'] = np.std(degree_seq)
        metrics['min_degree'] = min(degree_seq)
        metrics['max_degree'] = max(degree_seq)
        metrics['degree_variance'] = np.var(degree_seq)
        metrics['avg_in_degree'] = np.mean(in_degree_seq) if in_degree_seq else 0
        metrics['avg_out_degree'] = np.mean(out_degree_seq) if out_degree_seq else 0
        metrics['degree_assortativity'] = nx.degree_assortativity_coefficient(G_und)

        probs = np.bincount(degree_seq) / n_nodes
        probs = probs[probs > 0]
        metrics['degree_entropy'] = -np.sum(probs * np.log2(probs)) if len(probs) > 0 else 0
    else:
        metrics.update({
            'avg_degree': 0, 'median_degree': 0, 'std_degree': 0, 'min_degree': 0,
            'max_degree': 0, 'degree_variance': 0, 'avg_in_degree': 0, 'avg_out_degree': 0,
            'degree_assortativity': 0, 'degree_entropy': 0
        })

    if n_nodes > 0:
        metrics['avg_clustering'] = nx.average_clustering(G_und)
        metrics['transitivity'] = nx.transitivity(G_und)
        metrics['n_triangles'] = sum(nx.triangles(G_und).values()) // 3
    else:
        metrics['avg_clustering'] = 0
        metrics['transitivity'] = 0
        metrics['n_triangles'] = 0

    if nx.is_connected(G_und) and n_nodes > 1:
        metrics['diameter'] = nx.diameter(G_und)
        metrics['radius'] = nx.radius(G_und)
        metrics['avg_path_length'] = nx.average_shortest_path_length(G_und)

        ecc = nx.eccentricity(G_und)
        metrics['avg_eccentricity'] = np.mean(list(ecc.values()))
        metrics['max_eccentricity'] = max(ecc.values())
        metrics['min_eccentricity'] = min(ecc.values())
    else:
        if n_nodes > 0:
            largest_cc = max(nx.connected_components(G_und), key=len)
            subgraph = G_und.subgraph(largest_cc)
            if len(subgraph) > 1:
                metrics['diameter'] = nx.diameter(subgraph)
                metrics['radius'] = nx.radius(subgraph)
                metrics['avg_path_length'] = nx.average_shortest_path_length(subgraph)
                ecc = nx.eccentricity(subgraph)
                metrics['avg_eccentricity'] = np.mean(list(ecc.values()))
                metrics['max_eccentricity'] = max(ecc.values())
                metrics['min_eccentricity'] = min(ecc.values())
            else:
                metrics.update({'diameter': 0, 'radius': 0, 'avg_path_length': 0,
                               'avg_eccentricity': 0, 'max_eccentricity': 0, 'min_eccentricity': 0})
        else:
            metrics.update({'diameter': 0, 'radius': 0, 'avg_path_length': 0,
                           'avg_eccentricity': 0, 'max_eccentricity': 0, 'min_eccentricity': 0})

    if n_nodes > 0:
        deg_cent = nx.degree_centrality(G_und)
        metrics['avg_degree_centrality'] = np.mean(list(deg_cent.values()))
        metrics['max_degree_centrality'] = max(deg_cent.values())
        metrics['std_degree_centrality'] = np.std(list(deg_cent.values()))
        in_deg_cent = nx.in_degree_centrality(G)
        out_deg_cent = nx.out_degree_centrality(G)
        metrics['avg_in_degree_centrality'] = np.mean(list(in_deg_cent.values())) if in_deg_cent else 0
        metrics['avg_out_degree_centrality'] = np.mean(list(out_deg_cent.values())) if out_deg_cent else 0

        bet_cent = nx.betweenness_centrality(G_und)
        metrics['avg_betweenness'] = np.mean(list(bet_cent.values()))
        metrics['max_betweenness'] = max(bet_cent.values())
        metrics['std_betweenness'] = np.std(list(bet_cent.values()))

        if any('weight' in d for (_, _, d) in G.edges(data=True)):
            bet_cent_weighted = nx.betweenness_centrality(G_und, weight='weight')
            metrics['avg_betweenness_weighted'] = np.mean(list(bet_cent_weighted.values()))
            metrics['max_betweenness_weighted'] = max(bet_cent_weighted.values())
        else:
            metrics['avg_betweenness_weighted'] = metrics['avg_betweenness']
            metrics['max_betweenness_weighted'] = metrics['max_betweenness']

        if nx.is_connected(G_und):
            clo_cent = nx.closeness_centrality(G_und)
        else:
            clo_cent = nx.harmonic_centrality(G_und)
        metrics['avg_closeness'] = np.mean(list(clo_cent.values()))
        metrics['max_closeness'] = max(clo_cent.values())

        try:
            eig_cent = nx.eigenvector_centrality(G_und, max_iter=1000)
            metrics['avg_eigenvector'] = np.mean(list(eig_cent.values()))
            metrics['max_eigenvector'] = max(eig_cent.values())
        except:
            metrics['avg_eigenvector'] = 0
            metrics['max_eigenvector'] = 0

        try:
            katz_cent = nx.katz_centrality(G_und, alpha=0.1)
            metrics['avg_katz'] = np.mean(list(katz_cent.values()))
            metrics['max_katz'] = max(katz_cent.values())
        except:
            metrics['avg_katz'] = 0
            metrics['max_katz'] = 0

    else:
        null_cent = ['avg_degree_centrality', 'max_degree_centrality', 'std_degree_centrality',
                    'avg_in_degree_centrality', 'avg_out_degree_centrality', 'avg_betweenness',
                    'max_betweenness', 'std_betweenness', 'avg_betweenness_weighted', 'max_betweenness_weighted',
                    'avg_closeness', 'max_closeness', 'avg_eigenvector', 'max_eigenvector',
                    'avg_katz', 'max_katz']
        for c in null_cent:
            metrics[c] = 0

    path = [int(node) for node in path]
    seen = {}
    entry_node = None
    for idx, node in enumerate(path):
        if node in seen and idx - seen[node] > 1:
            entry_node = node
            break
        seen[node] = idx

    has_loop = entry_node is not None
    if has_loop:
        visits = 0
        prev = None
        for node in path:
            if node == entry_node and node != prev:
                visits += 1
            prev = node
        loop_count = max(visits - 1, 0)
    else:
        loop_count = 0

    adj = defaultdict(list)
    for u, v, w in zip(path, path[1:], distances):
        if u == v:
            continue
        adj[u].append((v, w))

    def dijkstra(u):
        dist = {u: 0}
        heap = [(0, u)]
        while heap:
            d, node = heapq.heappop(heap)
            for neighbor, weight in adj[node]:
                new_dist = d + weight
                if neighbor not in dist or new_dist < dist[neighbor]:
                    dist[neighbor] = new_dist
                    heapq.heappush(heap, (new_dist, neighbor))
        return dist

    all_distances = [dijkstra(node) for node in list(adj.keys())]
    diameter = max((max(distances.values()) for distances in all_distances), default=0)
    avg_path_length = \
        sum(sum(distances.values()) for distances in all_distances) / sum(len(distances) - 1 for distances in all_distances)

    undirected = defaultdict(set)
    for u, neighbors in adj.items():
        for v, _ in neighbors:
            undirected[u].add(v)
            undirected[v].add(u)
    clustering_sum, count_cc = 0, 0
    for node, nbrs in undirected.items():
        if len(nbrs) < 2:
            continue
        actual_edges = sum(1 for v in nbrs for w in nbrs if v < w and w in undirected[v])
        clustering_sum += actual_edges / (len(nbrs) * (len(nbrs) - 1) / 2)
        count_cc += 1
    avg_clustering = clustering_sum / count_cc if count_cc else 0
    N = len(undirected)
    K = sum(len(nbrs) for nbrs in undirected.values()) / N if N else 0
    C_rand = K / (N - 1) if N > 1 else 0
    L_rand = math.log(N) / math.log(K) if N > 1 and K > 1 else float('inf')
    clustering_norm = avg_clustering / C_rand if C_rand else 0
    path_length_norm = avg_path_length / L_rand if L_rand else 0
    small_world_index = clustering_norm / path_length_norm if path_length_norm else 0

    metrics['has_loop'] = has_loop
    metrics['loop_count'] = loop_count
    metrics['small_world_index'] = small_world_index

    return G, metrics

In [ ]:
import argparse
import json
from pathlib import Path
from typing import Iterable
import sys
import matplotlib.pyplot as plt

import numpy as np
from tqdm import tqdm


def ensure_dir(path: str | Path) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)


def load_npz(path: str | Path) -> dict[str, np.ndarray]:
    with np.load(path) as data:
        return {k: data[k] for k in data.files}


def process_embeddings_dir(
    emb_dir: Path,
    out_path: Path,
    graph_dir: Path
) -> int:
    ensure_dir(out_path)
    count = 0
    all_embeddings = []
    with out_path.open("w", encoding="utf-8") as out_f:
        for npz_path in tqdm(sorted(emb_dir.glob("*.npz")), desc=f"Graph features ({emb_dir.name})", unit="file"):
            pid = npz_path.stem
            arrays = load_npz(npz_path)
            X = arrays.get("X")
            if X is None or X.shape[0] < 2:
                continue
            all_embeddings.extend(X)

        print(len(all_embeddings))
        kmeans = build_global_clusters(all_embeddings)

        for npz_path in tqdm(sorted(emb_dir.glob("*.npz")), desc=f"Constructing graphs({emb_dir.name})", unit="file"):
            pid = npz_path.stem
            arrays = load_npz(npz_path)
            X = arrays.get("X")
            if X is None or X.shape[0] < 2:
                continue
            path, distances, node_avg_embeddings = build_graph(X, kmeans)
            G, metrics = analyze_graph_simple(path, distances, node_avg_embeddings)
            graph_path = graph_dir / f'{pid}.graphml'
            ensure_dir(graph_path)

            with graph_path.open("w", encoding="utf-8") as out_g:
              nx.write_gml(G, graph_path)

            row = {
                "id": pid,
                **metrics
            }
            out_f.write(json.dumps(row) + "\n")
            count += 1
    return count


def main() -> None:
    model = "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B"
    split = "trace"
    emb_dir = f"/content/data/math500_embed/{model}/trace"

    embed_root = "data/math500_embed"

    diag_root = "data/math_tda_diags"

    features_root = "data/math_graph"

    graph_dir = Path("graph_dir")

    out = None

    model_slug = model
    split = split

    emb_dir = Path(emb_dir) if emb_dir else Path(embed_root) / model_slug / split
    features_out = Path(out) if out else Path(features_root) / split / f"global_graph_{model_slug}_math_1000_clusters_all_graph_features.jsonl"

    if not emb_dir.exists():
        raise FileNotFoundError(f"Embeddings directory '{emb_dir}' not found")

    count = process_embeddings_dir(emb_dir, features_out, graph_dir)
    print(f"Wrote {split} graph features for {count} items to {features_out}")

if __name__ == "__main__":
    main()

In [ ]:
#!zip -r global_graphs_600_clusters_math500_Qwen2.5_32B.zip graph_dir

In [ ]:
#!zip -r global_graphs_100_clusters_math500_Qwen2.5_32B.zip graph_dir

In [ ]:
!zip -r global_graphs_200_clusters_math500_DeepSeek-R1-Distill-Qwen2.5-32B.zip graph_dir

In [ ]:
from google.colab import files

files.download('/content/global_graphs_200_clusters_math500_DeepSeek-R1-Distill-Qwen2.5-32B.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>